In [ ]:
# @markdown ---
# @markdown ## Install dependencies

!pip install -q gradio scikit-learn matplotlib pandas

# @markdown ---

In [ ]:
# @markdown ---
# @markdown ## Run 🧪 Supervised Lab

import tempfile
import warnings
warnings.filterwarnings("ignore")

import gradio as gr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons, make_circles, make_regression
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression, LinearRegression, BayesianRidge
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.kernel_ridge import KernelRidge

APP_STATE = {
    "df": None,
    "task_type": None,
    "feature_cols": None,
    "target_col": None,
    "pred_df": None,
    "model": None,
}

def _safe_int(x, default=42):
    try:
        return int(x)
    except Exception:
        return default

def _safe_float(x, default=0.05):
    try:
        return float(x)
    except Exception:
        return default

def _plot_dataset(X, y=None, title="Dataset preview"):
    fig, ax = plt.subplots(figsize=(6, 4))
    if X.shape[1] > 2:
        X_plot = PCA(n_components=2, random_state=42).fit_transform(X)
        xlabel, ylabel = "PC1", "PC2"
    elif X.shape[1] == 2:
        X_plot = X
        xlabel, ylabel = "x0", "x1"
    else:
        X_plot = np.column_stack([X[:, 0], np.zeros(len(X))])
        xlabel, ylabel = "x0", ""
    if y is None:
        ax.scatter(X_plot[:, 0], X_plot[:, 1], alpha=0.75, edgecolor="k")
    else:
        ax.scatter(X_plot[:, 0], X_plot[:, 1], c=y, alpha=0.8, edgecolor="k")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

def _infer_task(y):
    y_series = pd.Series(y)
    if pd.api.types.is_numeric_dtype(y_series):
        unique_count = y_series.nunique()
        if unique_count <= 20 and np.all(np.equal(np.mod(y_series.dropna(), 1), 0)):
            return "classification"
        return "regression"
    return "classification"

def _prepare_xy(df, target_col, selected_features, task_mode):
    work_df = df.copy()
    feature_cols = selected_features if selected_features else [c for c in work_df.columns if c != target_col]
    X = work_df[feature_cols].copy()
    y = work_df[target_col].copy()
    X = pd.get_dummies(X, drop_first=False)
    task_type = _infer_task(y) if task_mode == "auto" else task_mode
    if task_type == "classification" and not pd.api.types.is_numeric_dtype(y):
        y = pd.factorize(y)[0]
    return X, y, feature_cols, task_type

def _get_model(task_type, model_name):
    if task_type == "classification":
        models = {
            "LogisticRegression": LogisticRegression(max_iter=2000),
            "SVM": SVC(probability=True),
            "kNN": KNeighborsClassifier(),
            "Bayes": GaussianNB(),
            "Decision Tree": DecisionTreeClassifier(random_state=42),
        }
    else:
        models = {
            "LinearRegression": LinearRegression(),
            "SVR": SVR(),
            "kNR": KNeighborsRegressor(),
            "Bayesian Ridge": BayesianRidge(),
            "Kernel Ridge": KernelRidge(),
        }
    return models[model_name]

def _metrics_text(task_type, y_test, y_pred):
    if task_type == "classification":
        return (
            f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n"
            f"Precision: {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}\n"
            f"Recall: {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}\n"
            f"F1-score: {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}"
        )
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    return (
        f"MAE: {mean_absolute_error(y_test, y_pred):.4f}\n"
        f"RMSE: {rmse:.4f}\n"
        f"R²: {r2_score(y_test, y_pred):.4f}"
    )

def _plot_results(task_type, y_test, y_pred):
    fig, ax = plt.subplots(figsize=(6, 4))
    if task_type == "classification":
        cm = confusion_matrix(y_test, y_pred)
        im = ax.imshow(cm, cmap="Oranges")
        ax.set_title("Confusion Matrix")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center")
        fig.colorbar(im, ax=ax)
    else:
        ax.scatter(y_test, y_pred, alpha=0.75, edgecolor="k")
        min_v = min(np.min(y_test), np.min(y_pred))
        max_v = max(np.max(y_test), np.max(y_pred))
        ax.plot([min_v, max_v], [min_v, max_v], linestyle="--")
        ax.set_title("Predicted vs True")
        ax.set_xlabel("True")
        ax.set_ylabel("Predicted")
    plt.tight_layout()
    return fig

def generate_samples(n_samples, dataset_type, noise, angle_aniso, deg_reg, random_state, return_classes, n_features):
    rs = _safe_int(random_state, 42)
    noise = _safe_float(noise, 0.05)
    angle_aniso = _safe_float(angle_aniso, 110)
    deg_reg = _safe_int(deg_reg, 3)
    n_features = _safe_int(n_features, 2)

    if dataset_type == "blobs":
        X, y = make_blobs(
            n_samples=int(n_samples),
            centers=3,
            n_features=n_features,
            cluster_std=1.5,
            random_state=rs
        )
        task_type = "classification"
    elif dataset_type == "no_structure":
        X = np.random.rand(int(n_samples), n_features)
        y = None
        task = "none"
    elif dataset_type == "anisotropic":
        X, y = make_blobs(n_samples=int(n_samples), n_features=n_features, random_state=rs)
        t = np.tan(np.radians(angle_aniso))
        transformation = np.array(((1, t), (0, 1))).T
        X = np.dot(X, transformation)
        X += np.random.rand(int(n_samples), n_features) * noise * max(1.0, abs(X.min()))
        task = "classification"
    elif dataset_type == "varied_var":
        X, y = make_blobs(n_samples=int(n_samples), n_features=n_features, cluster_std=[1.0, 2.5, 0.5], random_state=rs)
        X += np.random.rand(int(n_samples), n_features) * noise * max(1.0, abs(X.min()))
        task = "classification"
    elif dataset_type == "moons":
        X, y = make_moons(n_samples=int(n_samples), noise=noise, random_state=rs)
        task_type = "classification"
    elif dataset_type == "circles":
        X, y = make_circles(n_samples=int(n_samples), noise=noise, factor=0.5, random_state=rs)
        task_type = "classification"
    elif dataset_type == "linear_reg":
        X, y = make_regression(
            n_samples=int(n_samples),
            n_features=int(n_features),
            noise=max(noise * 100, 1),
            random_state=rs
        )
        task_type = "regression"
    elif dataset_type == "nonlinear_reg":
        X, y = make_regression(n_samples=int(n_samples), n_features=int(n_features), noise=max(noise * 100, 1), random_state=rs)
        y = (y * 0.1) ** int(deg_reg)
        task_type = "regression"
    else:
        raise ValueError("Unknown dataset type")

    if dataset_type == "blobs" and X.shape[1] >= 2:
        theta = np.deg2rad(angle_aniso)
        rot = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
        X[:, :2] = X[:, :2] @ rot

    df = pd.DataFrame(X, columns=[f"x{i}" for i in range(X.shape[1])])
    target_col = "class" if task_type == "classification" else "target"
    df[target_col] = y

    APP_STATE["df"] = df
    APP_STATE["task_type"] = task_type
    APP_STATE["target_col"] = target_col
    APP_STATE["feature_cols"] = [c for c in df.columns if c != target_col]
    APP_STATE["pred_df"] = None
    APP_STATE["model"] = None

    plot = _plot_dataset(df[APP_STATE["feature_cols"]].values, df[target_col].values, f"Generated: {dataset_type}")
    info = f"Data shape: {df.shape}\nTask: {task_type}\nTarget column: {target_col}"
    target_choices = gr.update(choices=list(df.columns), value=target_col)
    feature_choices = gr.update(choices=list(df.columns), value=APP_STATE["feature_cols"])
    return df.head(20), plot, info, target_choices, feature_choices

def load_dataset(file_obj):
    if file_obj is None:
        return None, None, "Please upload a CSV file.", gr.update(), gr.update()
    df = pd.read_csv(file_obj.name)
    APP_STATE["df"] = df
    APP_STATE["pred_df"] = None
    APP_STATE["model"] = None

    target_col = df.columns[-1]
    task_type = _infer_task(df[target_col])
    APP_STATE["task_type"] = task_type
    APP_STATE["target_col"] = target_col
    APP_STATE["feature_cols"] = list(df.columns[:-1])

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    plot = None
    if len(numeric_cols) >= 2:
        X_plot = df[numeric_cols[:2]].values
        y_plot = df[target_col].values
        plot = _plot_dataset(X_plot, y_plot, "Loaded dataset preview")

    info = f"Loaded dataset with shape: {df.shape}\nDetected task: {task_type}\nDefault target: {target_col}"
    target_choices = gr.update(choices=list(df.columns), value=target_col)
    feature_choices = gr.update(choices=list(df.columns), value=APP_STATE["feature_cols"])
    return df.head(20), plot, info, target_choices, feature_choices

def update_model_choices(task_mode):
    task = APP_STATE["task_type"] if task_mode == "auto" and APP_STATE["task_type"] is not None else task_mode
    if task in [None, "auto"]:
        cls = ["LogisticRegression", "SVM", "kNN", "Bayes", "Decision Tree"]
        reg = ["LinearRegression", "SVR", "kNR", "Bayesian Ridge", "Kernel Ridge"]
        return gr.update(choices=cls + reg, value=cls[0])
    if task == "classification":
        cls = ["LogisticRegression", "SVM", "kNN", "Bayes", "Decision Tree"]
        return gr.update(choices=cls, value=cls[0])
    reg = ["LinearRegression", "SVR", "kNR", "Bayesian Ridge", "Kernel Ridge"]
    return gr.update(choices=reg, value=reg[0])

def run_training(task_mode, model_name, target_col, selected_features, test_size, random_state):
    if APP_STATE["df"] is None:
        return "No dataset available. Generate or load data first.", None, None, None

    df = APP_STATE["df"].copy()
    if target_col not in df.columns:
        return "Choose a valid target column.", None, None, None

    X, y, feature_cols, task_type = _prepare_xy(df, target_col, selected_features, task_mode)
    try:
        model = _get_model(task_type, model_name)
    except KeyError:
        return f"Model '{model_name}' is not valid for task '{task_type}'.", None, None, None

    rs = _safe_int(random_state, 42)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=float(test_size), random_state=rs,
        stratify=y if task_type == "classification" else None
    )

    use_scaler = model_name not in ["Bayes", "Decision Tree"]
    estimator = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ]) if use_scaler else model

    estimator.fit(X_train, y_train)
    y_pred = estimator.predict(X_test)

    metrics_text = _metrics_text(task_type, y_test, y_pred)
    results_plot = _plot_results(task_type, np.asarray(y_test), np.asarray(y_pred))

    pred_df = X_test.copy()
    pred_df["y_true"] = np.asarray(y_test)
    pred_df["prediction"] = np.asarray(y_pred)
    APP_STATE["pred_df"] = pred_df.reset_index(drop=True)
    APP_STATE["model"] = estimator
    APP_STATE["task_type"] = task_type
    APP_STATE["target_col"] = target_col
    APP_STATE["feature_cols"] = feature_cols

    with tempfile.NamedTemporaryFile(delete=False, suffix=".csv", mode="w", newline="") as tmp:
        APP_STATE["pred_df"].to_csv(tmp.name, index=False)
        out_path = tmp.name

    summary = (
        f"Training completed\n"
        f"Task: {task_type}\n"
        f"Model: {model_name}\n"
        f"Features used: {len(feature_cols)}\n"
        f"Train size: {len(X_train)} | Test size: {len(X_test)}"
    )
    return summary + "\n\n" + metrics_text, results_plot, APP_STATE["pred_df"].head(30), out_path


with gr.Blocks(title="Supervised Lab", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧪 Supervised Lab")

    with gr.Tab("Data Lab"):
        gr.Markdown("## Generate samples")
        with gr.Row():
            n_samples = gr.Number(value=300, label="n_samples")
            type_dataset = gr.Dropdown(
                choices=["blobs", "moons", "circles", "varied_var", "anisotropic", "no_structure", "linear_reg", "nonlinear_reg"],
                value="nonlinear_reg",
                label="type_dataset"
            )
            noise = gr.Slider(0.0, 1.0, value=0.05, step=0.01, label="noise")
        with gr.Row():
            angle_aniso = gr.Slider(0, 180, value=110, step=1, label="angle_aniso")
            deg_reg = gr.Slider(2, 6, value=3, step=1, label="deg_reg")
            random_state = gr.Number(value=4, label="random_state")
        with gr.Row():
            return_classes = gr.Checkbox(value=False, label="return_classes")
            # n_features = gr.Slider(2, 10, value=2, step=1, label="n_features (blobs only)")
            n_features = gr.Number(value=2, label="n_features (blobs only)")
        generate_btn = gr.Button("Generate samples", variant="primary")

        gr.Markdown("## Load a dataset")
        load_file = gr.File(file_types=[".csv"], label="Upload CSV")
        load_btn = gr.Button("Load dataset")

        data_info = gr.Textbox(label="Data info", lines=4)
        data_plot = gr.Plot(label="Preview")
        data_table = gr.Dataframe(label="Dataset preview")

    with gr.Tab("Model Lab"):
        gr.Markdown("## Classification and regression")
        with gr.Row():
            task_mode = gr.Dropdown(
                choices=["auto", "classification", "regression"],
                value="auto",
                label="Task mode"
            )
            model_name = gr.Dropdown(
                choices=["LogisticRegression", "SVM", "kNN", "Bayes", "Decision Tree"],
                value="LogisticRegression",
                label="Model"
            )
        with gr.Row():
            target_col = gr.Dropdown(choices=[], label="Target column")
            feature_cols = gr.Dropdown(choices=[], multiselect=True, label="Feature columns")
        with gr.Row():
            test_size = gr.Slider(0.1, 0.5, value=0.2, step=0.05, label="Test size")
            train_random_state = gr.Number(value=42, label="Random state")

    with gr.Tab("Run Training"):
        gr.Markdown("## Training and compute metrics")
        train_btn = gr.Button("Run training", variant="primary")
        train_summary = gr.Textbox(label="Training summary and metrics", lines=10)
        train_plot = gr.Plot(label="Results plot")
        pred_table = gr.Dataframe(label="Predictions preview")
        gr.Markdown("## Download predictions")
        download_btn = gr.File(label="CSV with prediction column")

    generate_btn.click(
        generate_samples,
        inputs=[n_samples, type_dataset, noise, angle_aniso, deg_reg, random_state, return_classes, n_features],
        outputs=[data_table, data_plot, data_info, target_col, feature_cols]
    )
    load_btn.click(
        load_dataset,
        inputs=[load_file],
        outputs=[data_table, data_plot, data_info, target_col, feature_cols]
    )
    task_mode.change(
        update_model_choices,
        inputs=[task_mode],
        outputs=[model_name]
    )
    train_btn.click(
        run_training,
        inputs=[task_mode, model_name, target_col, feature_cols, test_size, train_random_state],
        outputs=[train_summary, train_plot, pred_table, download_btn]
    )

demo.launch(share=True)

# @markdown ---

In [31]:
demo.close()